# 01 — Selected dataset setup and inspection
Downloads and inspects the dataset selected once in `src/notebook_config.py`. Test clips are counted but never decoded for evaluation.

In [1]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path('/content/phd_research')
if not REPO_ROOT.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/sanelehlabisa/phd_research.git', str(REPO_ROOT)], check=True)
else:
    checkout_dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
    if checkout_dirty:
        print('Existing Colab checkout has local changes; preserving them and skipping update.')
    else:
        subprocess.run(['git', 'pull', '--ff-only', 'origin', 'master'], cwd=REPO_ROOT, check=True)
IMPLEMENTATION_ROOT = REPO_ROOT / 'papers/001-journal-abnormal-activity-recognition/implementation'
bootstrap = subprocess.run([sys.executable, str(IMPLEMENTATION_ROOT / 'src/colab_bootstrap.py'), str(IMPLEMENTATION_ROOT / 'requirements.txt')])
if bootstrap.returncode == 75:
    raise SystemExit('Restart the Colab runtime, reconnect to the A100, and rerun from the top.')
bootstrap.check_returncode()
if str(IMPLEMENTATION_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION_ROOT))
import importlib
for module_name in [name for name in sys.modules if name == 'src' or name.startswith('src.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()


In [2]:
from src.notebook_utils import printable, prepare_selected_splits, validate_runtime
print(printable(validate_runtime(require_cuda=True)))


{
  "cuda_available": true,
  "cuda_device": "NVIDIA A100-SXM4-80GB",
  "cuda_version": "12.8",
  "cudnn_version": 91002,
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "python": "3.13.15",
  "pytorch": "2.9.1+cu128",
  "torchvision": "0.24.1+cu128"
}


In [3]:
prepared = prepare_selected_splits(IMPLEMENTATION_ROOT)
inventory_summary = {key: value for key, value in prepared['inventory'].items() if key != 'records'}
print(printable({
    'dataset': prepared['specification'],
    'inventory': inventory_summary,
    'split': prepared['split'],
    'test_count_locked': prepared['test_count_locked'],
}))


Using Colab cache for faster access to the 'violence-detection-dataset' dataset.
✅ 350 videos | mode=video | 2 classes: ['non-violent', 'violent']
{
  "dataset": "DiagnosticDataset(key='vdd', kaggle_handle='sanelehlabisa/violence-detection-dataset', accepted_classes=('non-violent', 'violent'), enabled=True)",
  "inventory": {
    "class_counts": {
      "non-violent": 120,
      "violent": 230
    },
    "classes": [
      "non-violent",
      "violent"
    ],
    "clip_count": 350,
    "corrupt_or_unreadable": [],
    "dataset_root": "/kaggle/input/violence-detection-dataset/violence-detection-dataset/violence-detection-dataset",
    "duplicate_risk": "Exact byte duplicates are listed; visually equivalent re-encodes are not detected.",
    "duration_seconds": {
      "count": 350,
      "maximum": 14.466666666666667,
      "median": 5.3,
      "minimum": 2.466666666666667
    },
    "evidence_role": "pipeline_learnability_diagnostic",
    "exact_duplicate_groups": [],
    "formats": {

In [4]:
training_index = prepared['train'].indices[0]
clip, label = prepared['dataset'][training_index]
print({'partition': 'train', 'shape': tuple(clip.shape), 'label': prepared['dataset'].class_names[label]})


{'partition': 'train', 'shape': (16, 3, 32, 32), 'label': 'non-violent'}


## Five reproducible training examples
These examples inspect data only; no model or test sample is used.

In [ ]:
import matplotlib.pyplot as plt
from src.notebook_utils import selected_training_examples
examples = selected_training_examples(IMPLEMENTATION_ROOT, count=5)
figure, axes = plt.subplots(1, 5, figsize=(18, 4))
for axis, example in zip(axes, examples):
    axis.imshow(example['preview_frame'].permute(1, 2, 0).numpy())
    axis.set_title(example['label'])
    axis.axis('off')
    print({key: value for key, value in example.items() if key != 'preview_frame'})
figure.tight_layout()
plt.show()
